In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
from torch.utils.data import DataLoader

# ==================== 1. 从你整理好的 model 和 dataset 库中导入核心组件 ====================
from dataset import CityscapesDataset
from model import DPAI_DeepLabV3P_Segmentation, CombinedLoss, CBAM_DeepLabV3P_Segmentation

# ==================== 2. 训练器定义 ====================
class SegmentationTrainer:
    # 注意把存放权重的目录改叫 checkpoints_deeplab 防止和其它的覆盖
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, scheduler, device, save_dir='checkpoints_deeplab'):
        self.model, self.train_loader, self.val_loader = model, train_loader, val_loader
        self.criterion, self.optimizer, self.scheduler, self.device = criterion, optimizer, scheduler, device
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        self.epoch_loss_history = []
        self.best_miou, self.epoch = 0, 0

    def train_epoch(self):
        self.model.train()
        total_loss, running_loss = 0, 0.0
        log_interval = 50

        for batch_idx, (images, targets, _, _) in enumerate(self.train_loader):
            images, targets = images.to(self.device), targets.to(self.device)
            outputs = self.model(images)
            loss = self.criterion(outputs, targets)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            loss_val = loss.item()
            total_loss += loss_val
            running_loss += loss_val

            if (batch_idx + 1) % log_interval == 0:
                avg_batch_loss = running_loss / log_interval
                print(f"  Epoch {self.epoch+1}, Batch {batch_idx+1}, Group Avg Loss: {avg_batch_loss:.4f}")
                running_loss = 0.0

        return total_loss / len(self.train_loader)

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        total_loss, confusion_matrix = 0, np.zeros((19, 19))
        for images, targets, _, _ in self.val_loader:
            images, targets = images.to(self.device), targets.to(self.device)
            outputs = self.model(images)
            total_loss += self.criterion(outputs, targets).item()
            preds = outputs.argmax(dim=1).cpu().numpy()

            for t, p in zip(targets.cpu().numpy(), preds):
                mask = (t != 255)
                label = 19 * t[mask].astype('int') + p[mask]
                confusion_matrix += np.bincount(label, minlength=19**2).reshape(19, 19)

        iu = np.diag(confusion_matrix) / (confusion_matrix.sum(axis=1) + confusion_matrix.sum(axis=0) - np.diag(confusion_matrix) + 1e-10)
        return total_loss / len(self.val_loader), np.mean(iu)

    def train(self, num_epochs):
        for epoch in range(num_epochs):
            self.epoch = epoch
            print(f"\nEpoch {epoch+1}/{num_epochs}")

            avg_epoch_loss = self.train_epoch()
            val_loss, val_miou = self.validate()

            self.epoch_loss_history.append({'epoch': epoch + 1, 'train_loss': avg_epoch_loss, 'val_loss': val_loss, 'miou': val_miou})

            print(f"--- Epoch {epoch+1} Summary ---")
            print(f"Average Train Loss: {avg_epoch_loss:.6f}")
            print(f"Average Val Loss:   {val_loss:.6f}")
            print(f"mIoU:               {val_miou:.4f}")

            if val_miou > self.best_miou:
                self.best_miou = val_miou
                # 兼容多卡的保存安全逻辑
                model_to_save = self.model.module if hasattr(self.model, 'module') else self.model
                torch.save(model_to_save.state_dict(), os.path.join(self.save_dir, 'DPAI_DeepLabV3P.pth'))
                print("🏆 创新高！保存最佳模型!")

# ==================== 3. 驱动脚本 ====================
def main():
    root_dir = r"E:\Laboratory files\code_project\city_data"  # 请核对你的路径
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. 实例化咱们组装好的 DeepLabV3+ 改进模型
    model = CBAM_DeepLabV3P_Segmentation(num_classes=19, pretrained=True)

    # 2. 自动检测和开启双路 4090 显卡加速
    if torch.cuda.device_count() > 1:
        print(f"🔥 检测到 {torch.cuda.device_count()} 张显卡，开启 nn.DataParallel 多卡加速！")
        model = nn.DataParallel(model)
    model = model.to(device)

    # 3. 准备数据
    train_dataset = CityscapesDataset(root_dir, 'train')
    val_dataset = CityscapesDataset(root_dir, 'val')

    print(f"训练集图片: {len(train_dataset)} | 验证集图片: {len(val_dataset)}")

    # 4090显卡显存大，你可以尝试把这里的 batch_size 改成 16 享受更快的速度
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8)

    # 4. 指定为我们在 model.py 里准备好的牛逼的 CombinedLoss (包含了 Dice 和 Focal)
    criterion = CombinedLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    # 5. 起飞！
    trainer = SegmentationTrainer(model, train_loader, val_loader, criterion, optimizer, None, device, save_dir='checkpoints_deeplab')
    trainer.train(num_epochs=50)

if __name__ == "__main__":
    main()
